# 🤖 Notebook 2: Silver Explanation Generation
## Smart Split Strategy: Gemini for Hateful + Templates for None

**Strategy**:
- ~1,000 hateful samples → Gemini API (diverse, high-quality)
- ~500 'None' class samples → instant rule-based templates
- **Total: ~1,500 explanations** (sufficient for LSTM decoder training)

**Runtime**: CPU-only, ~6 hours overnight

---
## 1. Setup

In [ ]:
!pip install -q google-genai

import os, json, time, glob, random
import pandas as pd
from tqdm.auto import tqdm
from google import genai
from google.genai import types

OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('✅ Environment ready')

In [ ]:
# ==========================================
# MULTI-SESSION CONFIGURATION
# ==========================================
# Run this notebook in TWO separate Kaggle sessions with TWO different API keys.
# In Session 1, set this to 'A'. In Session 2, set this to 'B'.
SESSION_PART = 'A'  # Change to 'B' in your second notebook!
# ==========================================\n

---
## 2. API Key

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    API_KEY = UserSecretsClient().get_secret(f"GEMINI_API_KEY_{SESSION_PART}")
except:
    API_KEY = os.environ.get(f"GEMINI_API_KEY_{SESSION_PART}", "")

if not API_KEY:
    raise ValueError("Set GEMINI_API_KEY_A and GEMINI_API_KEY_B in Kaggle Secrets.")

client = genai.Client(api_key=API_KEY)

# Quick test
test = client.models.generate_content(model='gemini-3.6-flash', contents='Say OK')
print(f'✅ API working. Model response: {test.text.strip()}')

---
## 3. Load & Split Dataset

In [ ]:
# Auto-detect
paths = glob.glob('/kaggle/input/**/train.json', recursive=True)
if not paths:
    raise FileNotFoundError('Attach TrainMultiHate dataset first.')
TRAIN_PATH = paths[0]
print(f'✅ Dataset: {TRAIN_PATH}')

df = pd.read_json(TRAIN_PATH)
df['type_of_hate'] = df['type_of_hate'].fillna('None')
df['target_of_hate'] = df['target_of_hate'].fillna('None')
print(f'Total samples: {len(df)}')

# Split into hateful vs non-hateful
df_hateful = df[df['type_of_hate'] != 'None'].copy()
df_none = df[df['type_of_hate'] == 'None'].copy()

# Sample 2000 hateful (stratified) + 1000 None TOTAL
HATEFUL_TARGET_TOTAL = 2000
NONE_TARGET_TOTAL = 1000

df_hateful_all = df_hateful.groupby('type_of_hate', group_keys=False).apply(
    lambda x: x.sample(min(len(x), max(1, int(HATEFUL_TARGET_TOTAL * len(x) / len(df_hateful)))),
                       random_state=42)
)
if len(df_hateful_all) > HATEFUL_TARGET_TOTAL:
    df_hateful_all = df_hateful_all.sample(n=HATEFUL_TARGET_TOTAL, random_state=42)

df_none_all = df_none.sample(n=min(NONE_TARGET_TOTAL, len(df_none)), random_state=42)

# SPLIT INTO PART A AND PART B
mid_hate = len(df_hateful_all) // 2
mid_none = len(df_none_all) // 2

if SESSION_PART == 'A':
    df_hateful_sample = df_hateful_all.iloc[:mid_hate]
    df_none_sample = df_none_all.iloc[:mid_none]
else:
    df_hateful_sample = df_hateful_all.iloc[mid_hate:]
    df_none_sample = df_none_all.iloc[mid_none:]

print(f'\nprint(f'\n--- Assigned to Session {SESSION_PART} ---')print(f'Hateful (for Gemini):   {len(df_hateful_sample)}')print(df_hateful_sample['type_of_hate'].value_counts())print(f'\n
print(f'\nNone (for templates):   {len(df_none_sample)}')print(f'\n
print(f'\nTotal for this session: {len(df_hateful_sample) + len(df_none_sample)}')\n


---
## 4. Generate Rule-Based Templates for 'None' Class (Instant)

In [ ]:
RESULTS_FILE = os.path.join(OUTPUT_DIR, f'silver_explanations_part{SESSION_PART}.json')

# Load existing checkpoint
processed_data = []
processed_indices = set()
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE, 'r', encoding='utf-8') as f:
        processed_data = json.load(f)
        processed_indices = {item['original_idx'] for item in processed_data}
    print(f'Loaded {len(processed_data)} existing explanations from checkpoint.')

# Generate templates for None class
none_templates = [
    "এই মন্তব্যে কোনো ঘৃণামূলক, আক্রমণাত্মক বা বিদ্বেষপূর্ণ ভাষা নেই, তাই এটি স্বাভাবিক মন্তব্য।",
    "মন্তব্যটিতে কোনো গালিগালাজ, বিদ্বেষ বা আপত্তিকর বিষয়বস্তু পাওয়া যায়নি।",
    "এই বাক্যে কোনো ঘৃণামূলক শব্দ বা আক্রমণাত্মক প্রকাশ নেই বলে এটি নিরপেক্ষ মন্তব্য হিসেবে শ্রেণিবদ্ধ।",
    "মন্তব্যটি সাধারণ মতামত প্রকাশ করে এবং এতে কোনো ঘৃণা বা সহিংসতামূলক উপাদান নেই।",
    "এই মন্তব্যে কাউকে লক্ষ্য করে কোনো অবমাননাকর বা বৈষম্যমূলক ভাষা ব্যবহার করা হয়নি।"
]

none_count = 0
for idx, row in df_none_sample.iterrows():
    if idx in processed_indices:
        continue
    processed_data.append({
        'original_idx': int(idx),
        'comment': row['comment'],
        'type_of_hate': 'None',
        'target_of_hate': 'None',
        'severity_of_hate': row['severity_of_hate'],
        'silver_explanation': random.choice(none_templates),
        'source': 'rule-based'
    })
    processed_indices.add(idx)
    none_count += 1

# Extreme Failsafe: Save immediately after templates
with open(RESULTS_FILE, 'w', encoding='utf-8') as f:
    json.dump(processed_data, f, ensure_ascii=False, indent=2)

print(f'✅ Generated {none_count} rule-based explanations for None class (instant).')
print(f'Total so far: {len(processed_data)}')\n

---
## 5. Generate Gemini Explanations for Hateful Samples

In [ ]:
PROMPT_TEMPLATE = """You are an expert Bengali linguist and hate speech annotator.
Analyze the following Bengali text and explain WHY it was classified with the given labels.

Text: "{comment}"
Labels:
- Hate Type: {type_of_hate}
- Target: {target_of_hate}
- Severity: {severity_of_hate}

Provide a concise 1-2 sentence explanation IN BENGALI highlighting the specific words
or tone that justify these labels. No filler.

Return ONLY valid JSON:
{{"explanation_bn": "<your Bengali explanation>"}}"""

def build_prompt(row):
    return PROMPT_TEMPLATE.format(
        comment=str(row['comment']),
        type_of_hate=str(row['type_of_hate']),
        target_of_hate=str(row['target_of_hate']),
        severity_of_hate=str(row['severity_of_hate'])
    )

# Count remaining
remaining = sum(1 for idx in df_hateful_sample.index if idx not in processed_indices)
print(f'Hateful samples remaining: {remaining}')
print(f'Already done: {len(df_hateful_sample) - remaining}')
print(f'Estimated time: ~{remaining * 10 / 3600:.1f} hours')
print(f'\nStarting Gemini generation...\n')

real_errors = 0
MAX_REAL_ERRORS = 15
SAVE_INTERVAL = 5 # EXTREME FAILSAFE: Save every 5 iterations!
gemini_count = 0

for idx, row in tqdm(df_hateful_sample.iterrows(), total=len(df_hateful_sample)):
    if idx in processed_indices:
        continue

    prompt = build_prompt(row)

    try:
        response = client.models.generate_content(
            model='gemini-3.6-flash',
            contents=prompt,
            config=types.GenerateContentConfig(temperature=0.3)
        )

        raw = response.text
        explanation = raw
        try:
            result = json.loads(raw)
            explanation = result.get('explanation_bn', raw)
        except json.JSONDecodeError:
            cleaned = raw.replace('```json', '').replace('```', '').strip()
            try:
                result = json.loads(cleaned)
                explanation = result.get('explanation_bn', cleaned)
            except json.JSONDecodeError:
                explanation = cleaned

        processed_data.append({
            'original_idx': int(idx),
            'comment': row['comment'],
            'type_of_hate': row['type_of_hate'],
            'target_of_hate': row['target_of_hate'],
            'severity_of_hate': row['severity_of_hate'],
            'silver_explanation': explanation,
            'source': 'gemini-3.6-flash'
        })
        processed_indices.add(idx)
        gemini_count += 1

        # Gentle rate limit: ~6 requests per minute
        time.sleep(10)

    except Exception as e:
        err_msg = str(e)
        if '429' in err_msg or 'RESOURCE_EXHAUSTED' in err_msg:
            # Rate limited — wait and retry, DON'T count as error
            print(f'Rate limited. Waiting 120s... (generated {gemini_count} so far)')
            time.sleep(120)
        else:
            print(f'Error at {idx}: {err_msg[:100]}')
            real_errors += 1
            time.sleep(15)
            if real_errors > MAX_REAL_ERRORS:
                print(f'Too many real errors ({real_errors}). Stopping.')
                break

    # Checkpoint save every 25 samples
    if gemini_count > 0 and gemini_count % SAVE_INTERVAL == 0:
        with open(RESULTS_FILE, 'w', encoding='utf-8') as f:
            json.dump(processed_data, f, ensure_ascii=False, indent=2)
        print(f'  💾 Checkpoint: {len(processed_data)} total saved')

# Final save
with open(RESULTS_FILE, 'w', encoding='utf-8') as f:
    json.dump(processed_data, f, ensure_ascii=False, indent=2)

print(f'\n✅ Done!')
print(f'Gemini explanations: {gemini_count}')
print(f'Total explanations:  {len(processed_data)}')
print(f'Real errors:         {real_errors}')
print(f'Saved to: {RESULTS_FILE}')\n

---
## 6. Quality Check

In [ ]:
# Show distribution
sources = {}
for item in processed_data:
    src = item.get('source', 'unknown')
    sources[src] = sources.get(src, 0) + 1
print('Source distribution:')
for k, v in sources.items():
    print(f'  {k}: {v}')

# Show 5 random Gemini-generated samples
gemini_samples = [s for s in processed_data if s.get('source') != 'rule-based']
if gemini_samples:
    print(f'\n--- 5 Random Gemini Samples ---')
    for s in random.sample(gemini_samples, min(5, len(gemini_samples))):
        print(f"\n{'='*60}")
        print(f"Comment: {s['comment'][:80]}...")
        print(f"Labels: {s['type_of_hate']} / {s['target_of_hate']} / {s['severity_of_hate']}")
        print(f"Explanation: {s['silver_explanation']}")